|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 2:</h2>|<h1>Batching<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: break it on purpose<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/'course/Part2_Batching/3_incidents'))

import math, time
import torch
import lab
from transformers import AutoModelForCausalLM, AutoTokenizer

In the incident file you went from a symptom to a cause. Here you go the other
way. You put one fault into a working batch, a working scheduler or a working
plan, and you watch what it does.

The routine for each exercise is the same:

1. Read the fault.
2. **Write your prediction in the cell.** Answer the four questions.
3. Run the cell.
4. Write down where your prediction was wrong. This line is the one that
   teaches you.

The four questions:

- **Crash?** Does it raise an error, or does it run?
- **When?** Which row, which token, which minute?
- **What?** What does the wrong result look like?
- **Which guard?** Which check would catch it?

The reference for a row of a batch is the same prompt run **alone**. Some
exercises compare in float32 with Qwen3-0.6B, because in bfloat16 a batch
changes the last bits of the arithmetic (Part 1, Ticket 5). In float32 a
correct batch gives the same tokens as the row alone.

Two exercises simulate a server with `lab.serve`. A simulation is the right
tool when the fault lives in the policy, not in the arithmetic.

This notebook needs a GPU with about 10 GB free.

In [ ]:
### run this cell

MODEL = 'Qwen/Qwen3-1.7B'
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).cuda().eval()
device = 'cuda'
TOKENS = 20

PROMPTS = ['The three largest cities in Japan are',
           'My favourite recipe for pancakes is',
           'The capital of France is',
           'Water boils at',
           'A good name for a black cat that likes to sleep in the sun all day is',
           'The first person to walk on the moon was',
           'The chemical symbol for gold is',
           'In 1492, Columbus']
lengths = [len(tokenizer(p).input_ids) for p in PROMPTS]
alone = [lab.greedy_alone(model, tokenizer, p, TOKENS) for p in PROMPTS]
print('prompt lengths:', lengths)

def compare(name, rows, reference, lengths):
  """One line for each row: the pads and the first different token."""
  print(name)
  for row, (got, want) in enumerate(zip(rows, reference)):
    print(f'  row {row}: {max(lengths) - lengths[row]:2d} pads, first difference at {lab.first_difference(got, want)}'
          f'   {tokenizer.decode(got)[:60]!r}')

# Exercise 1: pad on the right, read position -1

The batch pads on the right, and the loop reads the logits of the last
position of each row. Everything else is correct: the mask is there, and it
grows at each step.

This is Ticket 1 of the incident file. Predict which rows go wrong, and at
which token.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
@torch.inference_mode()
def fault_right_padding(prompts, max_tokens):
  tokenizer.padding_side = 'right'                                    # THE FAULT
  batch = tokenizer(prompts, return_tensors='pt', padding=True).to(device)
  mask = batch.attention_mask
  out = model(batch.input_ids, attention_mask=mask, use_cache=True)
  cache = out.past_key_values
  next_tokens = out.logits[:, -1].argmax(-1)
  rows = [[] for _ in prompts]
  for _ in range(max_tokens):
    for row, token in enumerate(next_tokens.tolist()):
      rows[row].append(token)
    mask = torch.cat([mask, mask.new_ones(len(prompts), 1)], dim=1)
    out = model(next_tokens[:, None], attention_mask=mask, past_key_values=cache, use_cache=True)
    cache = out.past_key_values
    next_tokens = out.logits[:, -1].argmax(-1)
  tokenizer.padding_side = 'left'
  return rows

compare('right padding', fault_right_padding(PROMPTS, TOKENS), alone, lengths)

# Exercise 2: the mask only in the prefill

The fix of Exercise 1: left padding, and the mask in the prefill. But the
decode steps do not pass the mask.

This is Ticket 2. Predict the first different token of each row. Run it in
float32 on Qwen3-0.6B, so that a correct row matches exactly.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
exact = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B', dtype=torch.float32).cuda().eval()
alone_fp32 = [lab.greedy_alone(exact, tokenizer, p, TOKENS) for p in PROMPTS]

@torch.inference_mode()
def fault_prefill_mask_only(m, prompts, max_tokens):
  tokenizer.padding_side = 'left'
  batch = tokenizer(prompts, return_tensors='pt', padding=True).to(device)
  out = m(batch.input_ids, attention_mask=batch.attention_mask, use_cache=True)
  cache = out.past_key_values
  next_tokens = out.logits[:, -1].argmax(-1)
  rows = [[] for _ in prompts]
  for _ in range(max_tokens):
    for row, token in enumerate(next_tokens.tolist()):
      rows[row].append(token)
    out = m(next_tokens[:, None], past_key_values=cache, use_cache=True)   # THE FAULT: no mask
    cache = out.past_key_values
    next_tokens = out.logits[:, -1].argmax(-1)
  return rows

compare('mask in the prefill only, float32', fault_prefill_mask_only(exact, PROMPTS, TOKENS), alone_fp32, lengths)
compare('the correct batch, float32', lab.batched_greedy(exact, tokenizer, PROMPTS, TOKENS), alone_fp32, lengths)
del exact; torch.cuda.empty_cache()

# Exercise 3: batch 80 against batch 64

Measure one decode step at several batch sizes, with 512 tokens of context in
each sequence. Compare each with the floor: the weights plus the KV cache of
every sequence, divided by the bandwidth of `./vc info`.

This is Ticket 3. Predict the gain in tokens/s from 32 to 64, and from 64 to
80.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
BANDWIDTH = 294e9            # replace with the streaming bandwidth of ./vc info on your card
CONTEXT = 512
weights = sum(p.numel() * p.element_size() for p in model.parameters())
kv_per_token = 2 * 28 * 8 * 128 * 2

steps = {}
previous = None
for batch in (1, 8, 32, 64, 80):
  ms = lab.decode_step_ms(model, batch, CONTEXT)
  steps[batch] = ms
  kv = batch * CONTEXT * kv_per_token
  floor = (weights + kv) / BANDWIDTH * 1000
  rate = batch / ms * 1000
  gain = f'  x{rate / previous:.2f}' if previous else ''
  previous = rate
  print(f'batch {batch:3d}: KV {kv / 1e9:4.2f} GB   step {ms:5.1f} ms   floor {floor:5.1f} ms '
        f'({floor / ms:.0%})   {rate:6.0f} tok/s{gain}')

# Exercise 4: the static batch with one long answer

Simulate the nightly job of Ticket 4: 10,000 summaries in static batches of
32. 96% of the answers have about 100 tokens, and 4% run to 2,000. A static
batch runs until its longest answer ends.

Predict the steps for each batch, and the fraction of the slots that do
useful work.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
import random
rng = random.Random(0)
answers = [2000 if rng.random() < 0.04 else rng.randint(60, 140) for _ in range(10_000)]

steps_static = useful = 0
for start in range(0, len(answers), 32):
  batch = answers[start:start + 32]
  steps_static += max(batch)                                          # THE FAULT: wait for the slowest
  useful += sum(batch)
batches = math.ceil(len(answers) / 32)
mean = sum(answers) / len(answers)
print(f'mean answer {mean:.0f} tokens; the plan: {batches} batches x {mean:.0f} steps = {batches * mean:,.0f} steps')
print(f'static batches: {steps_static:,} steps, {steps_static / (batches * mean):.1f}x the plan')
print(f'useful slots: {useful / (steps_static * 32):.0%}')
print(f'the same answers sorted by length first: '
      f'{sum(max(sorted(answers)[s:s + 32]) for s in range(0, len(answers), 32)):,} steps')

# Exercise 5: continuous batching that forgets the stop token

A simulated server with 32 slots and a backlog of 1,000 requests. The answers
have a median of about 180 tokens, and `max_tokens` is 1,024. The fault: a
request leaves its slot only at `max_tokens`, not when it emits the stop
token.

This is Ticket 5. Predict the useful fraction of the decoded tokens, and the
slowdown of the whole backlog.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
import random
rng = random.Random(1)
backlog = [(0.0, min(1024, max(8, int(rng.lognormvariate(math.log(180), 0.6)))), 1024) for _ in range(1000)]

for evict in (True, False):
  result = lab.serve(backlog, slots=32, step_ms=25, evict_on_stop=evict)   # THE FAULT when False
  print(f'evict on stop = {evict!s:5s}: backlog done after {result["end"] / 60:5.1f} min, '
        f'useful tokens {result["useful"] / result["total"]:.0%}')

# Exercise 6: 12 arrivals each second for 10 each second

A simulated server with 16 slots, 25 ms for each step, and answers of 64
tokens. So it finishes 16 / 0.025 / 64 = 10 requests each second. For 30
minutes, 12 requests arrive each second.

This is Ticket 6. Predict the time to the first token at minutes 5, 15 and
30.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
arrivals = lab.poisson_arrivals(rate=12, seconds=30 * 60)
requests = [(t, 64, 64) for t in arrivals]
result = lab.serve(requests, slots=16, step_ms=25)
for minute in (1, 5, 15, 30):
  window = [f - a for (a, _, _), f in zip(requests, result['first'])
            if minute * 60 - 30 <= a < minute * 60]
  print(f'arrived at minute {minute:2d}: time to the first token {sum(window) / len(window):6.1f} s')
print(f'predicted slope: (12 - 10) / 10 = 0.2 s of wait for each second of overload')

# Exercise 7: the pad token is the end-of-turn token

A multi-turn chat and a short chat share a batch. The code uses `<|im_end|>`
(151645) as the pad token, and it builds the mask as `ids != pad_id`. In a
Qwen3 chat, `<|im_end|>` also ends every turn.

This is Ticket 7. Predict the zeros of the mask of the long chat, and how its
answer changes. Run in float32.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
exact = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B', dtype=torch.float32).cuda().eval()
long_chat = [{'role': 'user', 'content': 'My name is Priya and I live in Lisbon.'},
             {'role': 'assistant', 'content': 'Nice to meet you, Priya!'},
             {'role': 'user', 'content': 'I have a dog called Bento.'},
             {'role': 'assistant', 'content': 'Bento is a lovely name for a dog.'},
             {'role': 'user', 'content': 'What is my name, where do I live, and what is my dog called?'}]
short_chat = [{'role': 'user', 'content': 'Say hello.'}]
texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True, enable_thinking=False)
         for c in (long_chat, short_chat)]
reference = [lab.greedy_alone(exact, tokenizer, t, 40) for t in texts]

PAD = 151645                                                          # <|im_end|>
token_lists = [tokenizer(t).input_ids for t in texts]
width = max(len(t) for t in token_lists)
ids = torch.tensor([[PAD] * (width - len(t)) + t for t in token_lists], device=device)
mask = (ids != PAD).long()                                            # THE FAULT
print('pads:', [width - len(t) for t in token_lists], '  zeros in the mask:', (mask == 0).sum(1).tolist())

@torch.inference_mode()
def run(ids, mask, max_tokens):
  out = exact(ids, attention_mask=mask, use_cache=True)
  cache, rows = out.past_key_values, [[] for _ in ids]
  next_tokens = out.logits[:, -1].argmax(-1)
  for _ in range(max_tokens):
    for row, token in enumerate(next_tokens.tolist()):
      rows[row].append(token)
    mask = torch.cat([mask, mask.new_ones(len(ids), 1)], dim=1)
    out = exact(next_tokens[:, None], attention_mask=mask, past_key_values=cache, use_cache=True)
    cache = out.past_key_values
    next_tokens = out.logits[:, -1].argmax(-1)
  return rows

rows = run(ids, mask, 40)
for name, got, want in zip(['long chat', 'short chat'], rows, reference):
  got = got[:got.index(PAD)] if PAD in got else got
  print(f'{name}: first difference at {lab.first_difference(got, want)}')
  print('   batch:', repr(tokenizer.decode(got)))
  print('   alone:', repr(tokenizer.decode(want)))
del exact; torch.cuda.empty_cache()

# Exercise 8: the fastest batch against the promise to each user

Take the step times of Exercise 3. The promise to the users is 25 tokens each
second for each user, so a step must take at most 40 ms.

This is Ticket 8. Predict the largest batch that keeps the promise, and how
much total throughput the promise costs.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
PROMISE = 25                                                          # tokens each second for each user
best = max(steps, key=lambda b: b / steps[b])
for batch, ms in steps.items():
  per_user = 1000 / ms
  flag = '' if per_user >= PROMISE else '   <- breaks the promise'
  print(f'batch {batch:3d}: {per_user:5.1f} tok/s for each user, {batch * per_user:6.0f} tok/s in total{flag}')
kept = max(b for b in steps if 1000 / steps[b] >= PROMISE)
print(f'the fastest batch: {best}. The largest batch that keeps the promise: {kept}, '
      f'which costs {1 - (kept / steps[kept]) / (best / steps[best]):.0%} of the total throughput')

# Exercise 9: three mystery batches

The module `mystery.py` holds three batched loops: `mystery_a`, `mystery_b`
and `mystery_c`. Each one takes a list of prompts and returns one answer for
each prompt, like `lab.batched_greedy`. Each one has one fault. **Do not open
the file.**

For each loop:

1. Run it on `PROMPTS`, and compare each row with `alone`.
2. Design a second experiment that makes the fault visible. Which variable do
   you change? The order of the prompts? Their number? The kind of prompt?
3. Write your diagnosis: the fault, and the experiment that proved it.
4. Only then, open `mystery.py` and check.

A hint about the method: each answer of one loop is a good answer, to some
prompt. One loop depends on the other prompts in the batch. One loop depends
on the size of the batch.

In [ ]:
from mystery import mystery_a, mystery_b, mystery_c

for name, loop in [('a', mystery_a), ('b', mystery_b), ('c', mystery_c)]:
  compare(f'mystery_{name}', loop(model, tokenizer, PROMPTS, TOKENS), alone, lengths)

**Your diagnosis**

- `mystery_a`: the fault, and the experiment that proves it:
- `mystery_b`: the fault, and the experiment that proves it:
- `mystery_c`: the fault, and the experiment that proves it:

# Your fingerprint table

Fill in this table from what you saw, not from what you predicted.

| Fault | Crash? | When it shows | What it looks like | The guard |
|---|---|---|---|---|
| right padding, read position -1 | | | | |
| the mask only in the prefill | | | | |
| a larger batch that reads more KV | | | | |
| a static batch with one long answer | | | | |
| no eviction at the stop token | | | | |
| arrivals above the service rate | | | | |
| the pad id is the end-of-turn id | | | | |
| the fastest batch breaks the promise | | | | |